# Sanskrit ASR Training
Fine-tuning `facebook/wav2vec2-large-xlsr-53` on the `ai4bharat/Kathbath` Sanskrit dataset using CTC.

## Step 0 — Authenticate with Hugging Face
Run this cell once to log in. The `ai4bharat/Kathbath` dataset is gated and requires approval:
https://huggingface.co/datasets/ai4bharat/Kathbath

In [ ]:
from huggingface_hub import login
login()  # Paste your HF token when prompted

## Step 1 — Imports & Environment

In [ ]:
import os
import re
import json
import torch
import jiwer
import numpy as np
from dataclasses import dataclass
from typing import Union

from datasets import load_dataset, Audio, DatasetDict
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer
)
from huggingface_hub import get_token

# Fix PyTorch CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Step 2 — Load Dataset

In [ ]:
hf_token = get_token()
if hf_token is None:
    raise RuntimeError("No Hugging Face token found. Run the login() cell above first.")

dataset = load_dataset("ai4bharat/Kathbath", "sanskrit", token=hf_token)

# Create validation split if missing
if "validation" not in dataset:
    split = dataset["train"].train_test_split(test_size=0.1, seed=42)
    dataset = DatasetDict({
        "train": split["train"],
        "validation": split["test"]
    })

# Reduce dataset size for memory safety (use 50%)
dataset["train"] = dataset["train"].select(range(len(dataset["train"]) // 2))
dataset["validation"] = dataset["validation"].select(range(len(dataset["validation"]) // 2))

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print(dataset)

## Step 3 — Audio Processing
Rename the audio column, cast it to 16 kHz, and extract the raw waveform array.

In [ ]:
dataset = dataset.rename_column("audio_filepath", "audio")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

def prepare_audio(batch):
    audio = batch["audio"]
    batch["speech"] = audio["array"]
    batch["sampling_rate"] = audio["sampling_rate"]
    return batch

columns_to_remove = [col for col in dataset["train"].column_names if col not in ("text", "speech", "sampling_rate")]
dataset = dataset.map(prepare_audio, remove_columns=columns_to_remove)
print(dataset)

## Step 4 — Text Normalization
Keep only Devanagari characters (U+0900–U+097F) and normalize whitespace.

In [ ]:
def normalize(text):
    text = re.sub(r"[^\u0900-\u097F ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

dataset = dataset.map(lambda x: {"text": normalize(x["text"])})
print("Sample text:", dataset["train"][0]["text"])

## Step 5 — Build Vocabulary & Processor
Extract unique Devanagari characters, build `vocab.json`, and create the `Wav2Vec2Processor`.

In [ ]:
def extract_chars(batch):
    all_text = " ".join(batch["text"])
    vocab = list(set(all_text))
    return {"vocab": [vocab]}

vocab_train = dataset["train"].map(
    extract_chars, batched=True,
    remove_columns=dataset["train"].column_names
)
vocab_val = dataset["validation"].map(
    extract_chars, batched=True,
    remove_columns=dataset["validation"].column_names
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_val["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}

vocab_dict["|"] = vocab_dict[" "]  # Space → word delimiter
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print(f"Vocabulary size: {len(vocab_dict)}")

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("Processor ready. Vocab size:", len(processor.tokenizer))

## Step 6 — Prepare Dataset (Features + Labels)

In [ ]:
def prepare_dataset(batch):
    # Encode audio
    inputs = processor(batch["speech"], sampling_rate=16000)
    batch["input_values"] = inputs.input_values[0]
    # Encode text labels (use processor(text=...) — NOT the deprecated as_target_processor)
    batch["labels"] = processor(text=batch["text"]).input_ids
    return batch

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    batched=False
)
print("Dataset prepared:", dataset)

## Step 7 — Load Model

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(processor.tokenizer),
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    ignore_mismatched_sizes=True,  # Required when vocab_size differs from checkpoint
)

# Freeze CNN feature encoder — only fine-tune the transformer layers
model.freeze_feature_encoder()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")

## Step 8 — Data Collator
Pads audio inputs and labels to the longest sequence in each batch.

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Replace padding token id with -100 so loss ignores padding
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor)
print("Data collator ready.")

## Step 9 — Metrics (WER)

In [ ]:
def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = jiwer.wer(label_str, pred_str)
    return {"wer": wer}

## Step 10 — Training Arguments & Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./sanskrit_asr",
    group_by_length=True,              # Batch similar-length sequences together → faster
    per_device_train_batch_size=1,     # Keep low to avoid OOM on GPU
    gradient_accumulation_steps=8,     # Effective batch size = 8
    eval_strategy="steps",
    num_train_epochs=10,
    fp16=True,
    gradient_checkpointing=True,       # Trade compute for memory
    learning_rate=3e-4,
    warmup_steps=500,
    max_grad_norm=1.0,
    logging_steps=100,
    save_steps=1000,
    eval_steps=1000,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=processor.feature_extractor,  # Replaces deprecated tokenizer=
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready. Starting training...")

## Step 11 — Train

In [ ]:
trainer.train()

## Step 12 — Save Model & Processor

In [ ]:
model.save_pretrained("./sanskrit_asr_model")
processor.save_pretrained("./sanskrit_asr_model")
print("Model and processor saved to ./sanskrit_asr_model")